In [ ]:
# PASSO 1 — Configurações (rode sempre primeiro após reiniciar)
import psycopg2, psycopg2.extras, os, glob, time
from tqdm.notebook import tqdm

PASTA_DADOS = r'G:\Meu Drive\Projetos.Python\Dados'
PG_HOST     = 'localhost'
PG_PORT     = 5432
PG_USER     = 'postgres'
PG_PASSWORD = 'Amorauva25@'
PG_DATABASE = 'rfb_cnpj'
BATCH_SIZE  = 10000

def get_conn():
    return psycopg2.connect(
        host=PG_HOST, port=PG_PORT,
        user=PG_USER, password=PG_PASSWORD,
        database=PG_DATABASE
    )

def listar(sufixo):
    return sorted([
        f for f in glob.glob(os.path.join(PASTA_DADOS, '*'))
        if sufixo.lower() in os.path.basename(f).lower()
    ])

print('OK — configurações carregadas!')

In [ ]:
# PASSO 2 — Recriar tabelas com campos flexíveis (VARCHAR)
conn = get_conn()
cur  = conn.cursor()
cur.execute('''
DROP TABLE IF EXISTS socios CASCADE;
DROP TABLE IF EXISTS estabelecimentos CASCADE;
DROP TABLE IF EXISTS empresas CASCADE;
DROP TABLE IF EXISTS cnaes CASCADE;
DROP TABLE IF EXISTS municipios CASCADE;

CREATE TABLE empresas (
    cnpj_basico  VARCHAR(20) PRIMARY KEY,
    razao_social TEXT,
    natureza_jur VARCHAR(10),
    porte        VARCHAR(10)
);

CREATE TABLE estabelecimentos (
    cnpj_basico    VARCHAR(20),
    cnpj_ordem     VARCHAR(10),
    cnpj_dv        VARCHAR(5),
    matriz_filial  VARCHAR(2),
    nome_fantasia  TEXT,
    situacao       VARCHAR(5),
    cnae_principal VARCHAR(10),
    uf             VARCHAR(5),
    cod_municipio  VARCHAR(10),
    ddd1           VARCHAR(5),
    telefone1      VARCHAR(20),
    ddd2           VARCHAR(5),
    telefone2      VARCHAR(20),
    email          TEXT,
    PRIMARY KEY (cnpj_basico, cnpj_ordem, cnpj_dv)
);

CREATE TABLE socios (
    id           SERIAL PRIMARY KEY,
    cnpj_basico  VARCHAR(20),
    tipo_socio   VARCHAR(2),
    nome_socio   TEXT,
    qualificacao VARCHAR(200),
    data_entrada VARCHAR(15)
);

CREATE TABLE cnaes (
    codigo    VARCHAR(10) PRIMARY KEY,
    descricao TEXT
);

CREATE TABLE municipios (
    codigo VARCHAR(10) PRIMARY KEY,
    nome   TEXT
);

CREATE INDEX idx_est_cnae   ON estabelecimentos(cnae_principal);
CREATE INDEX idx_est_uf     ON estabelecimentos(uf);
CREATE INDEX idx_est_sit    ON estabelecimentos(situacao);
CREATE INDEX idx_est_mun    ON estabelecimentos(cod_municipio);
CREATE INDEX idx_est_basico ON estabelecimentos(cnpj_basico);
CREATE INDEX idx_soc_basico ON socios(cnpj_basico);
CREATE INDEX idx_soc_nome   ON socios(nome_socio);
''')
conn.commit()
cur.close()
conn.close()
print('OK — tabelas recriadas com VARCHAR flexivel!')

In [ ]:
# PASSO 3 — Importar CNAEs e Municipios
def importar_simples(sufixo, tabela, colunas, parser):
    arqs = listar(sufixo)
    print(f'{tabela}: {len(arqs)} arquivo(s)')
    sql = f'INSERT INTO {tabela} ({colunas}) VALUES %s ON CONFLICT DO NOTHING'
    for arq in arqs:
        batch = []
        with open(arq, encoding='latin-1', errors='replace') as f:
            for linha in f:
                row = parser(linha.strip())
                if row:
                    batch.append(row)
        if batch:
            conn = get_conn()
            cur  = conn.cursor()
            psycopg2.extras.execute_values(cur, sql, batch)
            conn.commit()
            cur.close()
            conn.close()
            print(f'  OK {len(batch):,} registros — {os.path.basename(arq)}')

def parse_cnae(linha):
    for sep in [';', ',', '|']:
        if sep in linha:
            r = linha.split(sep)
            if len(r) >= 2:
                cod = r[0].strip().replace('"','').replace('.','').replace('-','').replace('/','') 
                desc = r[1].strip().replace('"','')
                if cod and desc:
                    return (cod, desc)
    return None

def parse_mun(linha):
    for sep in [';', ',', '|']:
        if sep in linha:
            r = linha.split(sep)
            if len(r) >= 2:
                cod  = r[0].strip().replace('"','')
                nome = r[1].strip().replace('"','')
                if cod and nome:
                    return (cod, nome)
    return None

importar_simples('CNAECSV',  'cnaes',      'codigo, descricao', parse_cnae)
importar_simples('MUNICCSV', 'municipios', 'codigo, nome',      parse_mun)

conn = get_conn()
cur  = conn.cursor()
cur.execute('SELECT COUNT(*) FROM cnaes')
print(f'\nTotal CNAEs     : {cur.fetchone()[0]:,}')
cur.execute('SELECT COUNT(*) FROM municipios')
print(f'Total Municipios: {cur.fetchone()[0]:,}')
cur.close()
conn.close()

In [ ]:
# PASSO 4 — Importar EMPRESAS (deixe rodando ~60 min)
SQL = 'INSERT INTO empresas (cnpj_basico,razao_social,natureza_jur,porte) VALUES %s ON CONFLICT DO NOTHING'
arqs = listar('EMPRECSV')
print(f'Empresas: {len(arqs)} arquivos')
total_geral = 0
t0 = time.time()

for arq in arqs:
    nome = os.path.basename(arq)
    print(f'\nLendo {nome}...')
    conn = get_conn()
    cur  = conn.cursor()
    batch = []
    total_arq = 0
    with open(arq, encoding='latin-1', errors='replace') as f:
        for linha in tqdm(f, desc=nome[:25], unit=' lin', mininterval=3):
            r = linha.strip().split(';')
            if len(r) < 2:
                continue
            batch.append((
                r[0].strip()[:20],
                r[1].strip()[:500],
                r[2].strip()[:10] if len(r) > 2 else '',
                r[5].strip()[:10] if len(r) > 5 else '',
            ))
            if len(batch) >= BATCH_SIZE:
                psycopg2.extras.execute_values(cur, SQL, batch, page_size=BATCH_SIZE)
                conn.commit()
                total_arq   += len(batch)
                total_geral += len(batch)
                batch = []
    if batch:
        psycopg2.extras.execute_values(cur, SQL, batch, page_size=BATCH_SIZE)
        conn.commit()
        total_arq   += len(batch)
        total_geral += len(batch)
    cur.close()
    conn.close()
    print(f'  OK {total_arq:,} empresas')

print(f'\nEMPRESAS concluido! Total: {total_geral:,} em {(time.time()-t0)/60:.1f} min')

In [ ]:
# PASSO 5 — Importar ESTABELECIMENTOS (deixe rodando ~90 min)
SQL = '''
INSERT INTO estabelecimentos
  (cnpj_basico,cnpj_ordem,cnpj_dv,matriz_filial,nome_fantasia,
   situacao,cnae_principal,uf,cod_municipio,
   ddd1,telefone1,ddd2,telefone2,email)
VALUES %s
ON CONFLICT DO NOTHING
'''
arqs = listar('ESTABELE')
print(f'Estabelecimentos: {len(arqs)} arquivos')
total_geral = 0
t0 = time.time()

for arq in arqs:
    nome = os.path.basename(arq)
    print(f'\nLendo {nome}...')
    conn = get_conn()
    cur  = conn.cursor()
    batch = []
    total_arq = 0
    with open(arq, encoding='latin-1', errors='replace') as f:
        for linha in tqdm(f, desc=nome[:25], unit=' lin', mininterval=3):
            r = linha.strip().split(';')
            if len(r) < 20:
                continue
            batch.append((
                r[0].strip()[:20],
                r[1].strip()[:10],
                r[2].strip()[:5],
                r[3].strip()[:2],
                r[4].strip()[:200],
                r[5].strip()[:5],
                r[11].strip()[:10],
                r[19].strip()[:5],
                r[20].strip()[:10],
                r[21].strip()[:5],
                r[22].strip()[:20],
                r[23].strip()[:5],
                r[24].strip()[:20],
                r[27].strip()[:200],
            ))
            if len(batch) >= BATCH_SIZE:
                psycopg2.extras.execute_values(cur, SQL, batch, page_size=BATCH_SIZE)
                conn.commit()
                total_arq   += len(batch)
                total_geral += len(batch)
                batch = []
    if batch:
        psycopg2.extras.execute_values(cur, SQL, batch, page_size=BATCH_SIZE)
        conn.commit()
        total_arq   += len(batch)
        total_geral += len(batch)
    cur.close()
    conn.close()
    print(f'  OK {total_arq:,} estabelecimentos')

print(f'\nESTABELECIMENTOS concluido! Total: {total_geral:,} em {(time.time()-t0)/60:.1f} min')

In [ ]:
# PASSO 6 — Importar SOCIOS (deixe rodando ~45 min)
SQL = 'INSERT INTO socios (cnpj_basico,tipo_socio,nome_socio,qualificacao,data_entrada) VALUES %s'
arqs = listar('SOCIOCSV')
print(f'Socios: {len(arqs)} arquivos')
total_geral = 0
t0 = time.time()

for arq in arqs:
    nome = os.path.basename(arq)
    print(f'\nLendo {nome}...')
    conn = get_conn()
    cur  = conn.cursor()
    batch = []
    total_arq = 0
    with open(arq, encoding='latin-1', errors='replace') as f:
        for linha in tqdm(f, desc=nome[:25], unit=' lin', mininterval=3):
            r = linha.strip().split(';')
            if len(r) < 5:
                continue
            batch.append((
                r[0].strip()[:20],
                r[1].strip()[:2],
                r[2].strip()[:200],
                r[4].strip()[:200] if len(r) > 4 else '',
                r[5].strip()[:15] if len(r) > 5 else '',
            ))
            if len(batch) >= BATCH_SIZE:
                psycopg2.extras.execute_values(cur, SQL, batch, page_size=BATCH_SIZE)
                conn.commit()
                total_arq   += len(batch)
                total_geral += len(batch)
                batch = []
    if batch:
        psycopg2.extras.execute_values(cur, SQL, batch, page_size=BATCH_SIZE)
        conn.commit()
        total_arq   += len(batch)
        total_geral += len(batch)
    cur.close()
    conn.close()
    print(f'  OK {total_arq:,} socios')

print(f'\nSOCIOS concluido! Total: {total_geral:,} em {(time.time()-t0)/60:.1f} min')

In [ ]:
# PASSO 7 — Verificar totais
conn = get_conn()
cur  = conn.cursor()
for tabela in ['empresas','estabelecimentos','socios','cnaes','municipios']:
    cur.execute(f'SELECT COUNT(*) FROM {tabela}')
    print(f'{tabela:<22} {cur.fetchone()[0]:>12,}')
cur.close()
conn.close()

In [ ]:
# PASSO 8 — Teste de prospeccao
import pandas as pd
conn = get_conn()
df = pd.read_sql('''
SELECT
    est.cnpj_basico || est.cnpj_ordem || est.cnpj_dv  AS cnpj,
    COALESCE(NULLIF(est.nome_fantasia,''), emp.razao_social) AS nome,
    est.cnae_principal,
    c.descricao   AS cnae_desc,
    m.nome        AS municipio,
    est.uf,
    '(' || est.ddd1 || ') ' || est.telefone1  AS telefone,
    est.email,
    s.nome_socio,
    s.qualificacao
FROM estabelecimentos est
JOIN empresas   emp ON emp.cnpj_basico  = est.cnpj_basico
JOIN cnaes      c   ON c.codigo         = est.cnae_principal
JOIN municipios m   ON m.codigo         = est.cod_municipio
LEFT JOIN socios s  ON s.cnpj_basico    = est.cnpj_basico
WHERE est.cnae_principal IN (\'8630501\',\'8630502\',\'4530701\',\'4530703\',\'4772500\')
  AND est.uf IN (\'PE\',\'CE\',\'GO\',\'MT\')
  AND est.situacao = \'02\'
LIMIT 200
''', conn)
conn.close()
print(f'{len(df)} registros encontrados!')
df.head(10)

In [ ]:
# PASSO 9 — Exportar para Excel
import pandas as pd
from datetime import date
conn = get_conn()
df = pd.read_sql('''
SELECT
    est.cnpj_basico || est.cnpj_ordem || est.cnpj_dv  AS cnpj,
    COALESCE(NULLIF(est.nome_fantasia,''), emp.razao_social) AS nome,
    emp.razao_social,
    est.cnae_principal,
    c.descricao   AS cnae_desc,
    m.nome        AS municipio,
    est.uf,
    CASE WHEN est.ddd1 != \'\'  THEN \'(\' || est.ddd1 || \') \' || est.telefone1 ELSE \'\'  END AS telefone1,
    CASE WHEN est.ddd2 != \'\'  THEN \'(\' || est.ddd2 || \') \' || est.telefone2 ELSE \'\'  END AS telefone2,
    est.email,
    s.nome_socio,
    s.qualificacao
FROM estabelecimentos est
JOIN empresas   emp ON emp.cnpj_basico  = est.cnpj_basico
JOIN cnaes      c   ON c.codigo         = est.cnae_principal
JOIN municipios m   ON m.codigo         = est.cod_municipio
LEFT JOIN socios s  ON s.cnpj_basico    = est.cnpj_basico
WHERE est.cnae_principal IN (\'8630501\',\'8630502\',\'4530701\',\'4530703\',\'4772500\')
  AND est.uf IN (\'PE\',\'CE\',\'GO\',\'MT\')
  AND est.situacao = \'02\'
ORDER BY est.uf, m.nome
''', conn)
conn.close()
saida = rf'G:\Meu Drive\Projetos.Python\prospeccao_{date.today()}.xlsx'
df.to_excel(saida, index=False)
print(f'{len(df):,} registros exportados!')
print(f'Arquivo: {saida}')